# 02-2. 微分・導関数 — 動かして確かめる

📖 解説: [`../02_derivatives.md`](../02_derivatives.md)

## このノートで触るもの
1. 3 形式での微分: SymPy / 数値差分 / JAX
2. 接線を可視化
3. 連鎖律の確認
4. 高階微分
5. 速度ベンチマーク (JAX vs SymPy)

> 🧭 **クイックナビ**: 📚 [ROOT (全体 TOP)](../../README.md) ・ 🏠 [章 TOP](../README.md) ・ 📖 [解説 md (02_derivatives.md)](../02_derivatives.md)

In [ ]:
import time
import numpy as np
import sympy as sp
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

%matplotlib inline

## 1. 3 形式での微分の比較

f(x) = x³ + 2x² − 5x + 7, x = 2 における微分は 3·4 + 2·4 − 5 = 15 のはず

In [ ]:
# 標準形式 1: SymPy
x = sp.Symbol('x')
f_sym = x**3 + 2*x**2 - 5*x + 7
df_sym = sp.diff(f_sym, x)
print('SymPy 記号微分:', df_sym)
print('SymPy at x=2:', df_sym.subs(x, 2))

In [ ]:
# 標準形式 2: 数値差分 (中央差分)
def f(x):
    return x**3 + 2*x**2 - 5*x + 7

def df_central(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)

print('数値差分 at x=2:', df_central(f, 2.0))

In [ ]:
# JAX 形式: 自動微分
def f_jax(x):
    return x**3 + 2*x**2 - 5*x + 7

df_jax = jax.grad(f_jax)
print('JAX at x=2:', df_jax(2.0))

## 2. 接線を可視化

In [ ]:
def plot_tangent(x0: float) -> None:
    """f(x) = x² の x0 における接線を描画."""
    f = lambda x: x**2
    df = lambda x: 2*x
    
    xs = np.linspace(-3, 3, 100)
    ys = f(xs)
    slope = df(x0)
    tangent_x = np.linspace(x0 - 2, x0 + 2, 10)
    tangent_y = f(x0) + slope * (tangent_x - x0)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(xs, ys, 'b-', label='f(x) = x²')
    ax.plot(tangent_x, tangent_y, 'r--', label=f'x={x0} での接線 (傾き {slope:.2f})')
    ax.plot(x0, f(x0), 'go', markersize=12)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=11)
    ax.set_title(f'x = {x0:.2f},  f(x) = {f(x0):.2f},  f\'(x) = {slope:.2f}')
    plt.show()

interact(plot_tangent, x0=FloatSlider(min=-2.5, max=2.5, step=0.1, value=1.0));

## 3. 連鎖律: y = sin(x²) を 3 形式で

In [ ]:
# SymPy
x = sp.Symbol('x')
y = sp.sin(x**2)
print('SymPy:', sp.diff(y, x), '  ← 2x·cos(x²)')
print('SymPy at x=1:', float(sp.diff(y, x).subs(x, 1)))

# 数値
f = lambda x: np.sin(x**2)
print('数値 at x=1:', df_central(f, 1.0))

# JAX
df = jax.grad(lambda x: jnp.sin(x**2))
print('JAX at x=1:', df(1.0))

# 手で計算: 2·1·cos(1²) = 2·cos(1) ≈ 1.0806
print('手計算:', 2 * np.cos(1.0))

## 4. 高階微分: f(x) = x⁵

In [ ]:
# JAX で grad を重ねがけ
f = lambda x: x**5
df = jax.grad(f)
d2f = jax.grad(df)
d3f = jax.grad(d2f)

x0 = 2.0
print(f'f(2)    = {f(x0):.1f}     (理論: 32)')
print(f'f\'(2)   = {df(x0):.1f}     (理論: 5·16 = 80)')
print(f'f\'\'(2)  = {d2f(x0):.1f}    (理論: 20·8 = 160)')
print(f'f\'\'\'(2) = {d3f(x0):.1f}    (理論: 60·4 = 240)')

## 5. 数値微分の精度の罠

In [ ]:
f = lambda x: x**3
x0 = 2.0
true_value = 12.0   # 3·x² at x=2 = 12

print(f'真の値: {true_value}')
print()
print(f'{"h":>10}  {"中央差分":>15}  {"絶対誤差":>15}')
for h in [1.0, 0.1, 0.01, 1e-4, 1e-6, 1e-8, 1e-10, 1e-13, 1e-15]:
    approx = df_central(f, x0, h)
    err = abs(approx - true_value)
    print(f'{h:>10.0e}  {approx:>15.10f}  {err:>15.2e}')
print()
print('→ h=1e-5 〜 1e-7 がスイートスポット。それより小さいと逆に悪化!')

## 6. ベンチマーク: 1万回の勾配計算

In [ ]:
n_iter = 10_000
x_vals = np.linspace(-1, 1, n_iter)

# 数値差分
f_np = lambda x: np.sin(x**2) + np.cos(x)
t0 = time.perf_counter()
for x_val in x_vals:
    df_central(f_np, x_val)
t_num = time.perf_counter() - t0

# JAX (jit 付き)
@jax.jit
def df_jit(x):
    return jax.grad(lambda x: jnp.sin(x**2) + jnp.cos(x))(x)
_ = df_jit(0.0).block_until_ready()  # warmup

t0 = time.perf_counter()
for x_val in x_vals:
    df_jit(jnp.float32(x_val)).block_until_ready()
t_jax = time.perf_counter() - t0

# vmap でバッチ計算
vmap_grad = jax.jit(jax.vmap(jax.grad(lambda x: jnp.sin(x**2) + jnp.cos(x))))
_ = vmap_grad(jnp.array(x_vals[:10])).block_until_ready()  # warmup
t0 = time.perf_counter()
_ = vmap_grad(jnp.array(x_vals)).block_until_ready()
t_vmap = time.perf_counter() - t0

print(f'数値差分 (Python loop):  {t_num*1000:>8.2f} ms')
print(f'JAX grad (Python loop):  {t_jax*1000:>8.2f} ms')
print(f'JAX grad + vmap (一発):  {t_vmap*1000:>8.2f} ms  ← 圧倒的!')

## まとめ
- 同じ微分を 3 形式 (SymPy / 数値 / JAX) で出せる
- 接線スライダーで「微分 = ある点の傾き」を体感
- 連鎖律は JAX の `grad` が自動で適用してくれる
- 数値微分は h を小さくしすぎないこと (1e-5 〜 1e-7 が安全)
- `jit + vmap + grad` の組み合わせが ML の本命

## 次へ
→ [`03_integrals.ipynb`](03_integrals.ipynb)  解説 → [`../03_integrals.md`](../03_integrals.md)

---

## 📍 ナビゲーション

| ← 前 | 🏠 章 TOP | 📚 全体 TOP | 次 → |
|---|---|---|---|
| [`01_limits.ipynb`](01_limits.ipynb) | [章 TOP](../README.md) | [📚 ROOT README](../../README.md) | [`03_integrals.ipynb`](03_integrals.ipynb) |